# 🔍 Comprensión y Análisis Exploratorio de Datos (EDA)

**Proyecto:** Modelo Predictivo de Comportamiento de Pago  
**Dataset:** Histórico de Créditos — Institución Financiera  
**Objetivo:** Predecir si un cliente pagará su crédito a tiempo (`Pago_atiempo`)  

---

Este notebook cubre:
1. Exploración inicial y caracterización de datos
2. Limpieza y corrección de tipos
3. Análisis univariable
4. Análisis bivariable (vs variable objetivo)
5. Análisis multivariable
6. Reglas de validación y transformaciones sugeridas


## 1. Importación de librerías y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Estilo de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
COLORS = {'azul': '#2196F3', 'rojo': '#F44336', 'verde': '#4CAF50', 'naranja': '#FF9800'}

print('✅ Configuración lista')

## 2. Carga del dataset

In [ ]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_PATH = os.path.join(BASE_DIR, 'Base_de_datos.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['fecha_prestamo'])
df_raw = df.copy()  # Copia de respaldo

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

---
## 3. EXPLORACIÓN INICIAL DE DATOS
### 3.1 Descripción general

In [ ]:
print('='*70)
print('DESCRIPCIÓN GENERAL DEL DATASET')
print('='*70)
print(f'Filas         : {df.shape[0]:,}')
print(f'Columnas      : {df.shape[1]}')
print(f'Memoria usada : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print()
df.info()

In [ ]:
df.head(5)

### 3.2 Caracterización de variables

Clasificamos las variables según su naturaleza estadística:
- **Numéricas continuas**: capital_prestado, salario_cliente, cuota_pactada, saldos, etc.
- **Numéricas discretas**: plazo_meses, edad_cliente, conteos de créditos
- **Categóricas nominales**: tipo_laboral (sin orden)
- **Categóricas ordinales**: tendencia_ingresos (Decreciente < Estable < Creciente)
- **Dicotómica**: Pago_atiempo (variable objetivo: 0/1)
- **Fecha**: fecha_prestamo
- **Identificadores/Codigos**: tipo_credito (código numérico categórico)

In [ ]:
caracterizacion = {
    'Variable': [],
    'Tipo_dato': [],
    'Tipo_estadístico': [],
    'Categoría': [],
    'Valores_únicos': []
}

for col in df.columns:
    n_uniq = df[col].nunique()
    dtype = str(df[col].dtype)
    
    if col == 'fecha_prestamo':
        tipo_est = 'Temporal'
        cat = 'Fecha'
    elif col == 'Pago_atiempo':
        tipo_est = 'Dicotómica'
        cat = 'Numérica/Binaria'
    elif col == 'tipo_credito':
        tipo_est = 'Nominal'
        cat = 'Categórica (código)'
    elif col == 'tendencia_ingresos':
        tipo_est = 'Ordinal'
        cat = 'Categórica ordinal'
    elif col == 'tipo_laboral':
        tipo_est = 'Nominal'
        cat = 'Categórica nominal'
    elif 'int' in dtype or 'float' in dtype:
        if n_uniq <= 20:
            tipo_est = 'Discreta'
            cat = 'Numérica discreta'
        else:
            tipo_est = 'Continua'
            cat = 'Numérica continua'
    else:
        tipo_est = 'Categórica'
        cat = 'Categórica'

    caracterizacion['Variable'].append(col)
    caracterizacion['Tipo_dato'].append(dtype)
    caracterizacion['Tipo_estadístico'].append(tipo_est)
    caracterizacion['Categoría'].append(cat)
    caracterizacion['Valores_únicos'].append(n_uniq)

df_caract = pd.DataFrame(caracterizacion)
print('CARACTERIZACIÓN DE VARIABLES')
print('='*70)
df_caract

### 3.3 Revisión y unificación de valores nulos

In [ ]:
# Primero identificamos valores que representan nulos pero no son NaN
# tendencia_ingresos tiene valores numéricos erróneos que deben tratarse como nulos
print('Valores únicos de tendencia_ingresos (muestra):')
print(df['tendencia_ingresos'].unique()[:20])
print()

# Los valores válidos son solo: 'Estable', 'Creciente', 'Decreciente', NaN
valores_validos_tendencia = ['Estable', 'Creciente', 'Decreciente']

# UNIFICACIÓN: convertir valores numéricos erróneos a NaN
df['tendencia_ingresos'] = df['tendencia_ingresos'].apply(
    lambda x: x if x in valores_validos_tendencia else np.nan
)

print('Después de la corrección:')
print(df['tendencia_ingresos'].value_counts(dropna=False))

In [ ]:
# Revisión completa de nulos
nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

print('VARIABLES CON VALORES NULOS')
print('='*50)
print(nulos)

# Visualización
fig, ax = plt.subplots(figsize=(10, 4))
nulos['Porcentaje (%)'].plot(kind='bar', ax=ax, color=COLORS['azul'], edgecolor='black')
ax.set_title('Porcentaje de Nulos por Variable', fontsize=14, fontweight='bold')
ax.set_ylabel('% de Nulos')
ax.set_xlabel('Variable')
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(nulos['Porcentaje (%)']):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print()
print('⚠️  INTERPRETACIÓN:')
print('   - tendencia_ingresos y promedio_ingresos_datacredito (~27% nulos): Posiblemente clientes')
print('     sin historial en Datacrédito. Se tratan como categoría "Sin información".')
print('   - saldo_mora_codeudor (~5.5%): Créditos sin codeudor. Se imputa con 0.')
print('   - saldo_principal (~3.8%) y saldos (~1.5%): Posible error en datos. Imputar con mediana.')
print('   - puntaje_datacredito (~0.06%): Muy pocos nulos, imputar con mediana.')

### 3.4 Eliminación de variables irrelevantes

In [ ]:
# Análisis de `fecha_prestamo`: es un identificador temporal, no predictivo directamente.
# Sin embargo, podemos derivar atributos útiles como: mes, trimestre, antigüedad del préstamo.
# La fecha en sí no se usa como feature crudo.
# DECISIÓN: extraer variables derivadas y luego eliminar la columna original.

# Verificación de varianza cero
print('Variables con varianza cercana a cero (posibles candidatas a eliminar):')
for col in df.select_dtypes(include=['number']).columns:
    if df[col].std() < 1e-6:
        print(f'  ⚠️  {col}: std = {df[col].std():.6f}')

# Verificación de columnas con un solo valor
single_val = [col for col in df.columns if df[col].nunique() == 1]
if single_val:
    print(f'Columnas con un único valor: {single_val}')
else:
    print('✅ No hay columnas con un único valor.')

# NOTA: puntaje tiene una moda muy dominante (95.227787) pero tiene valores negativos atípicos.
# Requiere revisión — no se elimina, se trata.
print(f"\nDistribución puntaje: {df['puntaje'].describe().to_dict()}")

### 3.5 Corrección de tipos de datos

In [ ]:
# ── Conversiones de tipo ──────────────────────────────────────────────

# 1. tipo_credito: código numérico → categórico
df['tipo_credito'] = df['tipo_credito'].astype('category')

# 2. tipo_laboral: objeto → categórico
df['tipo_laboral'] = df['tipo_laboral'].astype('category')

# 3. tendencia_ingresos: objeto → categórico ordinal
orden_tendencia = ['Decreciente', 'Estable', 'Creciente']
df['tendencia_ingresos'] = pd.Categorical(
    df['tendencia_ingresos'],
    categories=orden_tendencia,
    ordered=True
)

# 4. Pago_atiempo: int → bool/int (se mantiene como int para modelado)
df['Pago_atiempo'] = df['Pago_atiempo'].astype(int)

# 5. Extraer features temporales de fecha_prestamo
df['mes_prestamo'] = df['fecha_prestamo'].dt.month
df['trimestre_prestamo'] = df['fecha_prestamo'].dt.quarter
df['anio_prestamo'] = df['fecha_prestamo'].dt.year
df['dia_semana_prestamo'] = df['fecha_prestamo'].dt.dayofweek  # 0=Lunes

# 6. Edad_cliente: hay valores atípicos (123 años) → serán identificados en EDA
# 7. salario_cliente: hay valores extremos (2.2e10) → outliers

print('✅ Tipos de datos corregidos')
print()
df.dtypes

---
## 4. ANÁLISIS UNIVARIABLE
### 4.1 Descripción estadística general

In [ ]:
print('ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS')
df.describe().T

### 4.2 Variables numéricas: histogramas, boxplots y estadísticos

In [ ]:
def describe_numerica(df, col):
    """Estadísticos completos para variable numérica."""
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outliers = ((s < lower_fence) | (s > upper_fence)).sum()
    sk = stats.skew(s)
    ku = stats.kurtosis(s)
    
    if abs(sk) < 0.5:
        dist = 'Aproximadamente simétrica'
    elif sk > 2:
        dist = 'Asimétrica positiva (cola derecha)'
    elif sk < -2:
        dist = 'Asimétrica negativa (cola izquierda)'
    elif sk > 0:
        dist = 'Leve asimetría positiva'
    else:
        dist = 'Leve asimetría negativa'
    
    return {
        'Media': s.mean(), 'Mediana': s.median(), 'Moda': s.mode()[0],
        'Min': s.min(), 'Max': s.max(), 'Rango': s.max() - s.min(),
        'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'Varianza': s.var(), 'Desv_Std': s.std(),
        'Skewness': round(sk, 4), 'Kurtosis': round(ku, 4),
        'Outliers': outliers, 'Distribución': dist
    }

# Variables numéricas principales
num_cols = ['capital_prestado', 'plazo_meses', 'edad_cliente', 'salario_cliente',
            'cuota_pactada', 'puntaje', 'puntaje_datacredito', 'saldo_mora',
            'saldo_total', 'saldo_principal', 'cant_creditosvigentes', 'huella_consulta']

stats_df = pd.DataFrame([describe_numerica(df, c) for c in num_cols], index=num_cols)
print('ESTADÍSTICOS DESCRIPTIVOS COMPLETOS')
stats_df[['Media', 'Mediana', 'Moda', 'Desv_Std', 'Skewness', 'Kurtosis', 'Outliers', 'Distribución']]

In [ ]:
# Histogramas de variables numéricas clave
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    # Remover extremos para mejor visualización
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_viz = data[(data >= p1) & (data <= p99)]
    
    axes[i].hist(data_viz, bins=40, color=COLORS['azul'], edgecolor='white', alpha=0.8)
    axes[i].axvline(data.mean(), color=COLORS['rojo'], linestyle='--', label=f'Media: {data.mean():.0f}')
    axes[i].axvline(data.median(), color=COLORS['verde'], linestyle='--', label=f'Mediana: {data.median():.0f}')
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=8)

plt.suptitle('Distribución de Variables Numéricas (P1-P99)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n📌 INTERPRETACIÓN:')
print('   - capital_prestado, salario_cliente, saldo_total: Alta asimetría positiva (cola derecha).')
print('     Concentración en valores bajos con outliers extremos. Candidatos para transformación log.')
print('   - puntaje: Distribución bimodal/asimétrica. Mayoría en valor máximo (95.22 = clientes buenos).')
print('   - edad_cliente: Distribución aproximadamente normal, con outliers > 80 años (posible error).')
print('   - plazo_meses: Distribución discreta con picos en 6, 10, 12 meses (productos estándar).')

In [ ]:
# Boxplots para detectar outliers
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_viz = data[(data >= p1) & (data <= p99)]
    axes[i].boxplot(data_viz, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=COLORS['azul'], alpha=0.7))
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].tick_params(labelsize=8)

plt.suptitle('Boxplots — Variables Numéricas (P1-P99)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Variables categóricas: countplots y tablas

In [ ]:
cat_cols = ['tipo_credito', 'tipo_laboral', 'tendencia_ingresos']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    vc = df[col].value_counts(dropna=False)
    axes[i].bar(vc.index.astype(str), vc.values, color=COLORS['naranja'], edgecolor='black', alpha=0.8)
    axes[i].set_title(f'{col}\n(n={len(df[col].dropna()):,})', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Conteo')
    plt.setp(axes[i].get_xticklabels(), rotation=30, ha='right')
    for j, v in enumerate(vc.values):
        axes[i].text(j, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=8)

plt.suptitle('Distribución de Variables Categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 INTERPRETACIÓN:')
print('   - tipo_credito 4 domina (~70%). Hay un tipo 68 con muy pocos registros (posible outlier).')
print('   - tipo_laboral: 57% Empleado vs 43% Independiente. Dataset balanceado en esta dimensión.')
print('   - tendencia_ingresos: 27% nulos (sin historial Datacrédito). Resto: mayoría Estable/Creciente.')

In [ ]:
# Tablas pivote para categóricas
for col in cat_cols:
    print(f'\n📋 value_counts — {col}:')
    vc = df[col].value_counts(dropna=False)
    pct = df[col].value_counts(normalize=True, dropna=False) * 100
    tabla = pd.DataFrame({'Conteo': vc, '% del Total': pct.round(2)})
    print(tabla.to_string())

### 4.4 Variable objetivo: `Pago_atiempo`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

vc = df['Pago_atiempo'].value_counts()
colores_target = [COLORS['verde'], COLORS['rojo']]

# Pie chart
axes[0].pie(vc, labels=['Pagó a tiempo (1)', 'No pagó (0)'],
            autopct='%1.2f%%', colors=colores_target, startangle=90,
            explode=(0.05, 0.1))
axes[0].set_title('Proporción Variable Objetivo', fontweight='bold')

# Bar chart
axes[1].bar(['Pagó a tiempo (1)', 'No pagó (0)'], vc.values,
            color=colores_target, edgecolor='black', alpha=0.8)
axes[1].set_title('Conteo Variable Objetivo', fontweight='bold')
axes[1].set_ylabel('Cantidad')
for i, v in enumerate(vc.values):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.suptitle('⚠️  DESBALANCE DE CLASES: 95.2% vs 4.8%', fontsize=13, fontweight='bold', color='red')
plt.tight_layout()
plt.show()

print('\n⚠️  ALERTA CRÍTICA: Dataset altamente desbalanceado (imbalanced).')
print('    Estrategias recomendadas para el modelado:')
print('    1. Usar métricas ROC-AUC, F1, Precision-Recall (NO accuracy)')
print('    2. Aplicar SMOTE o RandomOverSampler en entrenamiento')
print('    3. Ajustar class_weight="balanced" en los modelos')
print('    4. Optimizar el umbral de decisión (threshold tuning)')

---
## 5. ANÁLISIS BIVARIABLE (vs Variable Objetivo)

In [ ]:
# Numéricas vs objetivo: comparación de distribuciones por clase
fig, axes = plt.subplots(3, 4, figsize=(22, 16))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in zip([0, 1], [COLORS['rojo'], COLORS['azul']]):
        data = df[df['Pago_atiempo'] == label][col].dropna()
        p1, p99 = data.quantile(0.01), data.quantile(0.99)
        data = data[(data >= p1) & (data <= p99)]
        axes[i].hist(data, bins=30, alpha=0.5, color=color,
                     label=f'Pagó={label}', density=True)
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución por Clase (Pago_atiempo)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 INTERPRETACIÓN:')
print('   - saldo_mora: Los que NO pagan tienen saldo_mora significativamente mayor.')
print('     Es la variable más discriminante visual.')
print('   - puntaje_datacredito: Clientes con bajo score tienden a no pagar.')
print('   - capital_prestado: Montos muy altos se asocian más con incumplimiento.')
print('   - edad_cliente: Distribuciones muy similares → bajo poder predictivo individual.')

In [ ]:
# Tabla pivote: medias por grupo
pivote_bivariable = df.groupby('Pago_atiempo')[num_cols].median().T
pivote_bivariable.columns = ['No Pagó (0)', 'Pagó a tiempo (1)']
pivote_bivariable['Diferencia'] = pivote_bivariable['Pagó a tiempo (1)'] - pivote_bivariable['No Pagó (0)']
pivote_bivariable['Ratio'] = (pivote_bivariable['Pagó a tiempo (1)'] / 
                               pivote_bivariable['No Pagó (0)'].replace(0, np.nan)).round(3)
print('MEDIANAS POR CLASE DE PAGO')
pivote_bivariable

In [ ]:
# Categóricas vs objetivo
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['Pago_atiempo'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=[COLORS['rojo'], COLORS['verde']],
            edgecolor='black', alpha=0.8)
    axes[i].set_title(f'{col} vs Pago_atiempo (%)', fontweight='bold')
    axes[i].set_ylabel('% dentro de la categoría')
    axes[i].legend(['No Pagó (0)', 'Pagó (1)'])
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].axhline(y=5, color='orange', linestyle='--', alpha=0.7, label='Promedio incumplimiento')

plt.suptitle('Tasa de Incumplimiento por Categoría', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 INTERPRETACIÓN:')
print('   - tipo_laboral: Independientes tienen ligeramente mayor tasa de incumplimiento.')
print('   - tendencia_ingresos: Decreciente → mayor incumplimiento (lógico). Creciente → menor.')
print('   - tipo_credito 68: Tasa de incumplimiento muy alta (outlier, pocos registros).')

---
## 6. ANÁLISIS MULTIVARIABLE

In [ ]:
# Matriz de correlación
num_df = df[num_cols + ['Pago_atiempo']].dropna()
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 8})
ax.set_title('Matriz de Correlación (Variables Numéricas)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlaciones con variable objetivo
corr_target = corr['Pago_atiempo'].drop('Pago_atiempo').sort_values(key=abs, ascending=False)
print('\nCORRELACIONES CON Pago_atiempo (ordenadas por valor absoluto):')
for col_name, val in corr_target.items():
    bar = '█' * int(abs(val) * 30)
    print(f'  {col_name:35s}: {val:+.4f}  {bar}')

print('\n📌 INTERPRETACIÓN:')
print('   - saldo_mora tiene la mayor correlación negativa con pagar a tiempo (-0.24).')
print('   - Correlaciones bajas en general → problema no lineal, buenos candidatos: Boosting, RF.')
print('   - Alta correlación saldo_total↔saldo_principal (0.96): multicolinealidad. Considerar quitar una.')
print('   - capital_prestado↔cuota_pactada (correlación alta): derivada directa, evaluar si aporta.')

In [ ]:
# Pairplot de variables más relevantes
top_vars = ['saldo_mora', 'puntaje_datacredito', 'capital_prestado', 'edad_cliente', 'Pago_atiempo']
df_pair = df[top_vars].dropna()

# Limitar outliers para visualización
for col in top_vars[:-1]:
    p1, p99 = df_pair[col].quantile(0.01), df_pair[col].quantile(0.99)
    df_pair = df_pair[(df_pair[col] >= p1) & (df_pair[col] <= p99)]

df_pair['Clase'] = df_pair['Pago_atiempo'].map({1: 'Pagó (1)', 0: 'No Pagó (0)'})

g = sns.pairplot(df_pair[top_vars[:-1] + ['Clase']], hue='Clase',
                 palette={'Pagó (1)': COLORS['azul'], 'No Pagó (0)': COLORS['rojo']},
                 diag_kind='kde', plot_kws={'alpha': 0.3, 's': 15})
g.fig.suptitle('Pairplot — Variables Clave por Clase', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📌 INTERPRETACIÓN:')
print('   - saldo_mora vs puntaje_datacredito: Se observa separación parcial de clases.')
print('     Clientes con alto saldo_mora y bajo score → mayor probabilidad de no pago.')
print('   - capital_prestado vs edad: Sin separación clara → interacción necesaria.')

In [ ]:
# Dispersión con hue para variables categóricas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: capital vs cuota coloreado por tipo_laboral
df_viz = df[(df['capital_prestado'] < df['capital_prestado'].quantile(0.99))].dropna(subset=['tipo_laboral'])
for lab, color in zip(['Empleado', 'Independiente'], [COLORS['azul'], COLORS['naranja']]):
    sub = df_viz[df_viz['tipo_laboral'] == lab]
    axes[0].scatter(sub['capital_prestado'], sub['cuota_pactada'],
                    alpha=0.3, s=10, label=lab, color=color)
axes[0].set_xlabel('Capital Prestado')
axes[0].set_ylabel('Cuota Pactada')
axes[0].set_title('Capital vs Cuota por Tipo Laboral', fontweight='bold')
axes[0].legend()

# Plot 2: puntaje_datacredito vs saldo_mora coloreado por Pago_atiempo
df_viz2 = df[(df['saldo_mora'] < df['saldo_mora'].quantile(0.99))].dropna(subset=['puntaje_datacredito'])
for pago, color, label in zip([1, 0], [COLORS['azul'], COLORS['rojo']], ['Pagó (1)', 'No Pagó (0)']):
    sub = df_viz2[df_viz2['Pago_atiempo'] == pago]
    axes[1].scatter(sub['puntaje_datacredito'], sub['saldo_mora'],
                    alpha=0.4, s=10, label=label, color=color)
axes[1].set_xlabel('Puntaje Datacrédito')
axes[1].set_ylabel('Saldo en Mora')
axes[1].set_title('Score vs Mora por Clase de Pago', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. REGLAS DE VALIDACIÓN Y TRANSFORMACIONES SUGERIDAS

In [ ]:
reglas = {
    'Variable': [
        'edad_cliente', 'salario_cliente', 'capital_prestado', 'plazo_meses',
        'puntaje_datacredito', 'puntaje', 'saldo_mora', 'tipo_credito',
        'tendencia_ingresos', 'tipo_laboral', 'saldo_mora_codeudor', 'promedio_ingresos_datacredito'
    ],
    'Regla_de_Validacion': [
        'Entre 18 y 90 años', 'Mayor o igual a 0', 'Mayor a 0',
        'Entre 1 y 120 meses', 'Entre -10 y 999', 'Entre -50 y 100',
        'Mayor o igual a 0', 'En [4, 6, 7, 9, 10, 68]',
        'En [Estable, Creciente, Decreciente, NaN]', 'En [Empleado, Independiente]',
        'Mayor o igual a 0 (NaN = sin codeudor)', 'Mayor o igual a 0'
    ],
    'Transformacion_Sugerida': [
        'Capping en P1-P99; winsorize', 'Log1p para reducir asimetría; capping extremo',
        'Log1p; capping P99', 'Mantener como discreta; posible binarización (<=12 meses)',
        'Imputar mediana donde nulo; mantener numérica', 'Capping valores negativos a 0',
        'Log1p (muchos ceros); indicador binario tiene_mora', 'OneHotEncoding',
        'OrdinalEncoding (Decreciente=0, Estable=1, Creciente=2) + indicador_sin_info',
        'BinaryEncoding (Empleado=1, Independiente=0)',
        'Imputar 0 (sin codeudor)', 'Log1p; imputar mediana'
    ]
}

df_reglas = pd.DataFrame(reglas)
print('REGLAS DE VALIDACIÓN Y TRANSFORMACIONES SUGERIDAS')
print('='*80)
df_reglas

In [ ]:
# Atributos derivados (feature engineering sugerido)
print('ATRIBUTOS ADICIONALES DERIVADOS SUGERIDOS')
print('='*70)
features_derivadas = pd.DataFrame({
    'Feature_Derivado': [
        'ratio_cuota_salario', 'ratio_deuda_ingreso', 'tiene_mora',
        'total_creditos', 'log_capital', 'log_salario',
        'antiguedad_prestamo_dias', 'ratio_mora_saldo_total',
        'puntaje_score_combinado', 'indicador_sin_datacredito'
    ],
    'Formula': [
        'cuota_pactada / salario_cliente',
        '(total_otros_prestamos + capital_prestado) / salario_cliente',
        '(saldo_mora > 0).astype(int)',
        'creditos_sectorFinanciero + creditos_sectorCooperativo + creditos_sectorReal',
        'log1p(capital_prestado)',
        'log1p(salario_cliente)',
        '(fecha_referencia - fecha_prestamo).dt.days',
        'saldo_mora / (saldo_total + 1)',
        '(puntaje + puntaje_datacredito/10) / 2',
        'tendencia_ingresos.isna().astype(int)'
    ],
    'Justificacion': [
        'Capacidad de pago — feature crítico en riesgo crediticio',
        'Nivel de apalancamiento total del cliente',
        'Indicador binario de mora activa — alta correlación con incumplimiento',
        'Exposición total en el sistema financiero',
        'Reduce asimetría del capital_prestado',
        'Reduce asimetría del salario',
        'Antigüedad desde el desembolso',
        'Severidad de la mora relativa',
        'Feature combinado de riesgo',
        'Bandera para clientes sin reporte en Datacrédito'
    ]
})
features_derivadas

In [ ]:
print('✅ Análisis EDA completado.')
print()
print('RESUMEN EJECUTIVO:')
print('─'*60)
print(f'  Registros analizados : {df.shape[0]:,}')
print(f'  Variables            : {df.shape[1]}')
print(f'  Tasa de incumplimiento: {(df["Pago_atiempo"]==0).mean()*100:.2f}%')
print(f'  Variables con nulos  : {df.isnull().any().sum()}')
print(f'  Variables con outliers: múltiples (ver boxplots)')
print()
print('HALLAZGOS CLAVE:')
print('  1. Dataset DESBALANCEADO (95% vs 5%) → usar SMOTE + métricas ajustadas')
print('  2. saldo_mora es la variable más discriminante')
print('  3. tendencia_ingresos tiene 27% nulos y datos sucios')
print('  4. Alta multicolinealidad saldo_total ↔ saldo_principal → eliminar una')
print('  5. Varios outliers extremos en salario y capital prestado')
print()
print('PRÓXIMO PASO: ft_engineering.py')